# Unconditional Latent Diffusion Model (LDM) — From Scratch

This notebook upgrades the pixel-space DDPM into a **Latent Diffusion Model** (LDM) — the architecture that underlies Stable Diffusion, minus text conditioning.

**The core idea:** instead of running the diffusion process on raw pixels (slow, expensive, most of the signal is high-frequency detail that doesn't matter much), we:

1. Train a **Variational Autoencoder (VAE)** that compresses images into a small, dense **latent space** (the "perceptual compression" stage).
2. Train a **UNet diffusion model entirely inside that latent space** (the "semantic compression" / generative stage).
3. At sampling time, denoise pure noise into a latent, then **decode it back to pixels** with the VAE decoder.

Because the latent grid is much smaller than the pixel grid (e.g. 4x downsampled in each spatial dimension = 16x fewer values, often with more channels but still far less total data once you account for redundant pixel-level detail), the UNet does dramatically less work per step, which is exactly why this architecture scales to large (512x512, 1024x1024) images — this is stage 1 and 2 of Stable Diffusion. (Stage 3, which we don't do here, is swapping the plain UNet for a *conditional* UNet with cross-attention to a text encoder.)

**Pipeline:**
```
Image (H x W x 3)  --[VAE Encoder]-->  Latent (h x w x c)  --[+noise, T steps]-->  Noisy latent
Noisy latent  --[UNet, T reverse steps]-->  Clean latent  --[VAE Decoder]-->  Image (H x W x 3)
```


In [ ]:
!pip install -q datasets

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import MNIST
from datasets import load_dataset
from tqdm.auto import tqdm

# --- CONFIGURATION ---
DATASET = "faces"  # "mnist" or "faces"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Diffusion Hyperparameters
TIMESTEPS = 300  # Standard is 1000, but 300 is much faster for Colab
BETA_START = 1e-4
BETA_END = 0.02

if DATASET == "mnist":
    CHANNELS = 1
    IMAGE_SIZE = 32       # Padded from 28x28 so dimensions divide cleanly
    VAE_BATCH_SIZE = 128
    VAE_EPOCHS = 5
    DIFFUSION_BATCH_SIZE = 128
    DIFFUSION_EPOCHS = 10
else:
    CHANNELS = 3
    IMAGE_SIZE = 64        # Bumped vs the pixel-space DDPM (32) — this is exactly the
                            # resolution where latent diffusion starts paying off, since
                            # the UNet never has to look at the full 64x64x3 grid directly.
    VAE_BATCH_SIZE = 64
    VAE_EPOCHS = 8
    DIFFUSION_BATCH_SIZE = 64
    DIFFUSION_EPOCHS = 15

# Latent space config (the VAE's compressed representation)
LATENT_CHANNELS = 4        # Same as Stable Diffusion's AutoencoderKL
VAE_DOWNSAMPLE_FACTOR = 4  # Two stride-2 downsamples inside the encoder -> 4x smaller per side
LATENT_SIZE = IMAGE_SIZE // VAE_DOWNSAMPLE_FACTOR

VAE_LR = 1e-4
DIFFUSION_LR = 2e-4
KL_WEIGHT = 1e-6  # Tiny weight on the KL term: we want a near-deterministic, high-fidelity
                   # autoencoder (good reconstructions), just regularized enough that the
                   # latent space is smooth and well-scaled for the diffusion model.

print(f"Device: {DEVICE}")
print(f"Pixel space: {IMAGE_SIZE}x{IMAGE_SIZE}x{CHANNELS} = {IMAGE_SIZE*IMAGE_SIZE*CHANNELS} values")
print(f"Latent space: {LATENT_SIZE}x{LATENT_SIZE}x{LATENT_CHANNELS} = {LATENT_SIZE*LATENT_SIZE*LATENT_CHANNELS} values")
print(f"Compression ratio: {(IMAGE_SIZE*IMAGE_SIZE*CHANNELS) / (LATENT_SIZE*LATENT_SIZE*LATENT_CHANNELS):.1f}x")


## 1. Dataset

Same data pipeline as the pixel-space DDPM notebook — MNIST or CelebA faces, scaled to `[-1, 1]`.

In [ ]:
if DATASET == "mnist":
    transform = transforms.Compose([
        transforms.Pad(2),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])  # Scale to [-1, 1]
    ])
    dataset = MNIST(root="./data", train=True, download=True, transform=transform)

elif DATASET == "faces":
    transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])
    print("Downloading Face Dataset (CelebA)...")
    hf_dataset = load_dataset("nielsr/CelebA-faces", split="train")

    class HFFaceDataset(torch.utils.data.Dataset):
        def __init__(self, hf_ds, transform):
            self.hf_ds = hf_ds
            self.transform = transform
        def __len__(self):
            return len(self.hf_ds)
        def __getitem__(self, idx):
            img = self.hf_ds[idx]['image']
            return self.transform(img), 0

    dataset = HFFaceDataset(hf_dataset, transform)

vae_dataloader = DataLoader(dataset, batch_size=VAE_BATCH_SIZE, shuffle=True, drop_last=True)
diffusion_dataloader = DataLoader(dataset, batch_size=DIFFUSION_BATCH_SIZE, shuffle=True, drop_last=True)
print(f"Dataset loaded. {len(dataset)} images. VAE batches/epoch: {len(vae_dataloader)}, Diffusion batches/epoch: {len(diffusion_dataloader)}")


## 2. Stage 1 — The Autoencoder (Perceptual Compression)

This is the piece that's new compared to the pixel-space DDPM. It's a **KL-regularized VAE** (same family as Stable Diffusion's `AutoencoderKL`):

- **Encoder**: downsamples the image by 4x in each spatial dimension via strided convolutions, and outputs a `mean` and `logvar` per latent pixel (so the latent is a distribution, not a single point).
- **Reparameterization trick**: `z = mean + std * eps` — lets us backprop through a sampling step.
- **Decoder**: mirrors the encoder, upsampling `z` back to a full-resolution image.
- **Loss** = reconstruction loss (pixel-level L1) + a *small* weight on the KL divergence between the latent distribution and a standard normal.

The KL term is deliberately tiny (`KL_WEIGHT = 1e-6`). We're not trying to build a great generative model out of the VAE itself — a full-weight VAE would blur outputs to satisfy the KL term. We just want the latent space to be *reasonably well-behaved* (roughly unit-scale, no crazy outliers) so that the diffusion model — trained separately, next — has an easy target to model. All the actual "generation" work happens in the diffusion stage.

In [ ]:
class ResnetBlock(nn.Module):
    """Pre-activation residual block: GroupNorm -> SiLU -> Conv, twice, plus a skip connection."""
    def __init__(self, in_ch, out_ch, groups=8):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(groups, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        h = self.conv1(F.silu(self.norm1(x)))
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=1)
    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)
    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        return self.conv(x)


class Encoder(nn.Module):
    """Image -> (mean, logvar) over the latent distribution."""
    def __init__(self, in_ch, base_ch=64, latent_ch=4, ch_mults=(1, 2, 4)):
        super().__init__()
        self.conv_in = nn.Conv2d(in_ch, base_ch, 3, padding=1)
        chs = [base_ch * m for m in ch_mults]
        layers = []
        prev = base_ch
        for i, ch in enumerate(chs):
            layers.append(ResnetBlock(prev, ch))
            if i != len(chs) - 1:          # downsample between stages, not after the last one
                layers.append(Downsample(ch))
            prev = ch
        self.blocks = nn.ModuleList(layers)
        self.norm_out = nn.GroupNorm(8, prev)
        self.conv_out = nn.Conv2d(prev, 2 * latent_ch, 3, padding=1)  # 2x for mean + logvar

    def forward(self, x):
        x = self.conv_in(x)
        for layer in self.blocks:
            x = layer(x)
        x = self.conv_out(F.silu(self.norm_out(x)))
        mean, logvar = torch.chunk(x, 2, dim=1)
        return mean, logvar


class Decoder(nn.Module):
    """Latent z -> reconstructed image."""
    def __init__(self, out_ch, base_ch=64, latent_ch=4, ch_mults=(1, 2, 4)):
        super().__init__()
        chs = [base_ch * m for m in reversed(ch_mults)]
        self.conv_in = nn.Conv2d(latent_ch, chs[0], 3, padding=1)
        layers = []
        prev = chs[0]
        for i, ch in enumerate(chs):
            layers.append(ResnetBlock(prev, ch))
            if i != len(chs) - 1:
                layers.append(Upsample(ch))
            prev = ch
        self.blocks = nn.ModuleList(layers)
        self.norm_out = nn.GroupNorm(8, prev)
        self.conv_out = nn.Conv2d(prev, out_ch, 3, padding=1)

    def forward(self, z):
        x = self.conv_in(z)
        for layer in self.blocks:
            x = layer(x)
        x = self.conv_out(F.silu(self.norm_out(x)))
        return torch.tanh(x)  # match the [-1, 1] normalization used on the input images


class VAE(nn.Module):
    def __init__(self, in_ch=3, base_ch=64, latent_ch=4, ch_mults=(1, 2, 4)):
        super().__init__()
        self.encoder = Encoder(in_ch, base_ch, latent_ch, ch_mults)
        self.decoder = Decoder(in_ch, base_ch, latent_ch, ch_mults)

    def encode(self, x):
        mean, logvar = self.encoder(x)
        logvar = torch.clamp(logvar, -30.0, 20.0)  # numerical safety
        return mean, logvar

    def reparameterize(self, mean, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mean + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mean, logvar = self.encode(x)
        z = self.reparameterize(mean, logvar)
        recon = self.decode(z)
        return recon, mean, logvar


def vae_loss(recon, target, mean, logvar, kl_weight=KL_WEIGHT):
    recon_loss = F.l1_loss(recon, target)
    kl_loss = -0.5 * torch.mean(torch.sum(1 + logvar - mean.pow(2) - logvar.exp(), dim=[1, 2, 3]))
    return recon_loss + kl_weight * kl_loss, recon_loss, kl_loss


In [ ]:
vae = VAE(in_ch=CHANNELS, base_ch=64, latent_ch=LATENT_CHANNELS, ch_mults=(1, 2, 4)).to(DEVICE)
vae_optimizer = torch.optim.Adam(vae.parameters(), lr=VAE_LR)

n_params = sum(p.numel() for p in vae.parameters())
print(f"VAE parameters: {n_params/1e6:.2f}M")
print(f"Starting VAE training on {DEVICE}...")

for epoch in range(VAE_EPOCHS):
    epoch_recon, epoch_kl = 0.0, 0.0
    pbar = tqdm(vae_dataloader, desc=f"VAE Epoch {epoch+1}/{VAE_EPOCHS}")

    vae.train()
    for images, _ in pbar:
        images = images.to(DEVICE)

        recon, mean, logvar = vae(images)
        loss, recon_loss, kl_loss = vae_loss(recon, images, mean, logvar)

        vae_optimizer.zero_grad()
        loss.backward()
        vae_optimizer.step()

        epoch_recon += recon_loss.item()
        epoch_kl += kl_loss.item()
        pbar.set_postfix(recon=recon_loss.item(), kl=kl_loss.item())

    print(f"VAE Epoch {epoch+1}: recon_loss={epoch_recon/len(vae_dataloader):.4f}, "
          f"kl_loss={epoch_kl/len(vae_dataloader):.2f}")


### Sanity check: VAE reconstructions

Before training the diffusion model on top of this latent space, confirm the autoencoder actually reconstructs images well. Top row = originals, bottom row = reconstructions.

In [ ]:
@torch.no_grad()
def show_reconstructions(n=8):
    vae.eval()
    images, _ = next(iter(vae_dataloader))
    n = min(n, images.shape[0])  # guard against a batch size smaller than n
    images = images[:n].to(DEVICE)
    recon, mean, logvar = vae(images)

    images = (images.clamp(-1, 1) + 1) / 2
    recon = (recon.clamp(-1, 1) + 1) / 2

    fig, axes = plt.subplots(2, n, figsize=(2 * n, 4))
    for i in range(n):
        for row, tensor, label in [(0, images, "Original"), (1, recon, "Reconstruction")]:
            img = tensor[i].cpu().permute(1, 2, 0)
            ax = axes[row, i]
            if CHANNELS == 1:
                ax.imshow(img.squeeze(), cmap="gray")
            else:
                ax.imshow(img.numpy())
            ax.axis("off")
            if i == 0:
                ax.set_ylabel(label)
    plt.tight_layout()
    plt.show()

show_reconstructions()


### Freezing the VAE and computing a latent scaling factor

Once the autoencoder is trained, we freeze it completely — it's now just a fixed "compression codec" for the diffusion model to work through. It is never updated again.

One more practical detail borrowed directly from Stable Diffusion: raw VAE latents usually don't have unit variance, and diffusion models are tuned assuming inputs are roughly `N(0, 1)`. So we measure the empirical std of the latents and rescale by `1 / std` (a single scalar, `SCALING_FACTOR`) before running diffusion on them, and divide it back out before decoding.

In [ ]:
vae.eval()
for p in vae.parameters():
    p.requires_grad = False

@torch.no_grad()
def compute_scaling_factor(n_batches=10):
    stds = []
    for i, (images, _) in enumerate(vae_dataloader):
        if i >= n_batches:
            break
        images = images.to(DEVICE)
        mean, logvar = vae.encode(images)
        z = vae.reparameterize(mean, logvar)
        stds.append(z.std().item())
    return 1.0 / (sum(stds) / len(stds))

SCALING_FACTOR = compute_scaling_factor()
print(f"Latent scaling factor: {SCALING_FACTOR:.4f}")


## 3. Stage 2 — Diffusion in Latent Space

This is the same DDPM machinery as the pixel-space notebook (linear noise scheduler, sinusoidal timestep embeddings, MSE loss on predicted noise) — the only thing that changes is *what* it operates on: latents of shape `(LATENT_CHANNELS, LATENT_SIZE, LATENT_SIZE)` instead of images of shape `(CHANNELS, IMAGE_SIZE, IMAGE_SIZE)`.

Since the latent grid is small, we can also afford **self-attention** inside the UNet's bottleneck (and one resolution level up), which is normally too expensive at full pixel resolution — this is one of the reasons latent diffusion models produce sharper, more globally coherent images than pixel-space DDPMs of similar compute budget.

In [ ]:
class LinearNoiseScheduler:
    def __init__(self, num_timesteps, beta_start, beta_end, device="cuda"):
        self.num_timesteps = num_timesteps
        self.device = device

        self.betas = torch.linspace(beta_start, beta_end, num_timesteps, device=device)
        self.alphas = 1. - self.betas
        self.alpha_cum_prod = torch.cumprod(self.alphas, dim=0)

        self.sqrt_alpha_cum_prod = torch.sqrt(self.alpha_cum_prod)
        self.sqrt_one_minus_alpha_cum_prod = torch.sqrt(1. - self.alpha_cum_prod)

    def _extract(self, a, t, x_shape):
        """Extracts values for the current timestep and reshapes for broadcasting"""
        batch_size = t.shape[0]
        out = a.gather(-1, t)
        return out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))

    def add_noise(self, original, noise, t):
        sqrt_alpha_cum_prod = self._extract(self.sqrt_alpha_cum_prod, t, original.shape)
        sqrt_one_minus_alpha_cum_prod = self._extract(self.sqrt_one_minus_alpha_cum_prod, t, original.shape)
        return sqrt_alpha_cum_prod * original + sqrt_one_minus_alpha_cum_prod * noise

    def sample_prev_timestep(self, xt, noise_pred, t):
        sqrt_alpha_cum_prod = self._extract(self.sqrt_alpha_cum_prod, t, xt.shape)
        sqrt_one_minus_alpha_cum_prod = self._extract(self.sqrt_one_minus_alpha_cum_prod, t, xt.shape)
        betas_t = self._extract(self.betas, t, xt.shape)
        alphas_t = self._extract(self.alphas, t, xt.shape)

        # Predict x0 (in latent space this time)
        x0 = (xt - (sqrt_one_minus_alpha_cum_prod * noise_pred)) / sqrt_alpha_cum_prod
        x0 = torch.clamp(x0, -4., 4.)  # latents aren't bounded to [-1, 1] like pixels, so clamp loosely

        # Calculate mean
        mean = xt - ((betas_t * noise_pred) / sqrt_one_minus_alpha_cum_prod)
        mean = mean / torch.sqrt(alphas_t)

        if t[0] == 0:
            return mean, x0
        else:
            alpha_cum_prod_t = self._extract(self.alpha_cum_prod, t, xt.shape)
            alpha_cum_prod_t_prev = self._extract(self.alpha_cum_prod, t - 1, xt.shape)

            variance = (1. - alpha_cum_prod_t_prev) / (1. - alpha_cum_prod_t) * betas_t
            sigma = torch.sqrt(variance)
            z = torch.randn_like(xt)
            return mean + sigma * z, x0


In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings


class ResBlockT(nn.Module):
    """Residual block with a timestep-embedding injection (FiLM-style additive bias)."""
    def __init__(self, in_ch, out_ch, time_dim, groups=8):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_mlp = nn.Linear(time_dim, out_ch)
        self.norm2 = nn.GroupNorm(groups, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_mlp(F.silu(t_emb))[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class SelfAttention(nn.Module):
    """Standard QKV self-attention over spatial positions. Cheap here because the
    latent grid is small (e.g. 16x16 or 8x8), unlike full pixel resolution."""
    def __init__(self, ch, groups=8):
        super().__init__()
        self.norm = nn.GroupNorm(groups, ch)
        self.qkv = nn.Conv2d(ch, ch * 3, 1)
        self.proj = nn.Conv2d(ch, ch, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h).reshape(B, 3, C, H * W).permute(1, 0, 3, 2)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = torch.softmax((q @ k.transpose(-1, -2)) / math.sqrt(C), dim=-1)
        out = attn @ v
        out = out.permute(0, 2, 1).reshape(B, C, H, W)
        return x + self.proj(out)


class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, use_attn=False, downsample=True):
        super().__init__()
        self.res = ResBlockT(in_ch, out_ch, time_dim)
        self.attn = SelfAttention(out_ch) if use_attn else nn.Identity()
        self.down = nn.Conv2d(out_ch, out_ch, 4, 2, 1) if downsample else nn.Identity()

    def forward(self, x, t):
        x = self.res(x, t)
        x = self.attn(x)
        skip = x                 # skip connection is taken BEFORE downsampling
        x = self.down(x)
        return x, skip


class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, use_attn=False, upsample=True):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch, 4, 2, 1) if upsample else nn.Identity()
        self.res = ResBlockT(in_ch + out_ch, out_ch, time_dim)
        self.attn = SelfAttention(out_ch) if use_attn else nn.Identity()

    def forward(self, x, skip, t):
        x = self.up(x)            # upsample FIRST so spatial size matches the skip connection
        x = torch.cat([x, skip], dim=1)
        x = self.res(x, t)
        x = self.attn(x)
        return x


class LatentUNet(nn.Module):
    """Same overall shape as the pixel-space UNet, but operates on LATENT_CHANNELS-channel
    tensors at LATENT_SIZE resolution, and adds self-attention at the two smallest scales."""
    def __init__(self, latent_ch=4, base_ch=128, time_dim=256):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )
        self.conv_in = nn.Conv2d(latent_ch, base_ch, 3, padding=1)

        self.down1 = DownBlock(base_ch, base_ch, time_dim, use_attn=False)               # full latent res
        self.down2 = DownBlock(base_ch, base_ch * 2, time_dim, use_attn=True)             # 1/2 latent res
        self.down3 = DownBlock(base_ch * 2, base_ch * 4, time_dim, use_attn=True, downsample=False)  # 1/4 latent res

        self.mid_res1 = ResBlockT(base_ch * 4, base_ch * 4, time_dim)
        self.mid_attn = SelfAttention(base_ch * 4)
        self.mid_res2 = ResBlockT(base_ch * 4, base_ch * 4, time_dim)

        self.up3 = UpBlock(base_ch * 4, base_ch * 4, time_dim, use_attn=True, upsample=False)
        self.up2 = UpBlock(base_ch * 4, base_ch * 2, time_dim, use_attn=True)
        self.up1 = UpBlock(base_ch * 2, base_ch, time_dim, use_attn=False)

        self.norm_out = nn.GroupNorm(8, base_ch)
        self.conv_out = nn.Conv2d(base_ch, latent_ch, 3, padding=1)

    def forward(self, x, timestep):
        t = self.time_mlp(timestep)
        x = self.conv_in(x)

        x, s1 = self.down1(x, t)
        x, s2 = self.down2(x, t)
        x, s3 = self.down3(x, t)

        x = self.mid_res1(x, t)
        x = self.mid_attn(x)
        x = self.mid_res2(x, t)

        x = self.up3(x, s3, t)
        x = self.up2(x, s2, t)
        x = self.up1(x, s1, t)

        x = self.conv_out(F.silu(self.norm_out(x)))
        return x


### Training the latent diffusion UNet

Each step: encode a batch of images to latents with the frozen VAE (`no_grad`), rescale by `SCALING_FACTOR`, add noise, and train the UNet to predict that noise — exactly the DDPM objective, just operating on a 16x (or more) smaller tensor per image.

In [ ]:
unet = LatentUNet(latent_ch=LATENT_CHANNELS, base_ch=128).to(DEVICE)
scheduler = LinearNoiseScheduler(num_timesteps=TIMESTEPS, beta_start=BETA_START, beta_end=BETA_END, device=DEVICE)

diffusion_optimizer = torch.optim.Adam(unet.parameters(), lr=DIFFUSION_LR)
loss_fn = nn.MSELoss()

n_params = sum(p.numel() for p in unet.parameters())
print(f"Latent UNet parameters: {n_params/1e6:.2f}M")
print(f"Starting latent diffusion training on {DEVICE}...")

for epoch in range(DIFFUSION_EPOCHS):
    epoch_loss = 0.0
    pbar = tqdm(diffusion_dataloader, desc=f"Diffusion Epoch {epoch+1}/{DIFFUSION_EPOCHS}")

    unet.train()
    for images, _ in pbar:
        images = images.to(DEVICE)

        # 1. Encode to latents with the FROZEN vae (no grad flows into the VAE)
        with torch.no_grad():
            mean, logvar = vae.encode(images)
            latents = vae.reparameterize(mean, logvar) * SCALING_FACTOR

        # 2. Sample random timesteps
        t = torch.randint(0, TIMESTEPS, (latents.shape[0],), device=DEVICE).long()

        # 3. Add noise (in latent space)
        noise = torch.randn_like(latents)
        noisy_latents = scheduler.add_noise(latents, noise, t)

        # 4. Predict noise
        noise_pred = unet(noisy_latents, t)

        # 5. Loss & optimize
        loss = loss_fn(noise_pred, noise)

        diffusion_optimizer.zero_grad()
        loss.backward()
        diffusion_optimizer.step()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    print(f"Diffusion Epoch {epoch+1} Loss: {epoch_loss / len(diffusion_dataloader):.4f}")


## 4. Sampling — Noise to Latent to Image

The reverse diffusion loop is identical in structure to the pixel-space DDPM — start from Gaussian noise and iteratively denoise. The only difference is that we run it on a `(LATENT_CHANNELS, LATENT_SIZE, LATENT_SIZE)` tensor, and add one final step: **decode** the resulting latent back into a full-resolution image with the frozen VAE decoder (after un-scaling by `SCALING_FACTOR`).

In [ ]:
@torch.no_grad()
def sample_images(n_samples=16):
    unet.eval()
    vae.eval()

    # Start from pure noise, IN LATENT SPACE
    zt = torch.randn((n_samples, LATENT_CHANNELS, LATENT_SIZE, LATENT_SIZE)).to(DEVICE)

    # Reverse diffusion loop from T-1 down to 0, entirely in latent space
    for i in tqdm(reversed(range(TIMESTEPS)), total=TIMESTEPS, desc="Sampling (latent space)"):
        t = torch.full((n_samples,), i, device=DEVICE, dtype=torch.long)
        noise_pred = unet(zt, t)
        zt, z0_pred = scheduler.sample_prev_timestep(zt, noise_pred, t)

    # Undo the latent scaling, then decode back to pixels — this is the one extra step
    # versus pixel-space DDPM, and it's what lets the UNet above stay cheap.
    latents = zt / SCALING_FACTOR
    images = vae.decode(latents)

    # Scale from [-1, 1] to [0, 1] for matplotlib
    images = (images.clamp(-1, 1) + 1) / 2
    return images


print("Generating images...")
generated = sample_images(16)

fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for i, ax in enumerate(axes.flat):
    img = generated[i].cpu().permute(1, 2, 0)
    if CHANNELS == 1:
        ax.imshow(img.squeeze(), cmap="gray")
    else:
        ax.imshow(img.numpy())
    ax.axis("off")
plt.tight_layout()
plt.suptitle("Unconditional Latent Diffusion — Samples", y=1.02)
plt.show()


## What's still missing before this is "Stable Diffusion"

This notebook covers the two unconditional building blocks — the autoencoder and the latent UNet. The remaining pieces that turn this into text-to-image Stable Diffusion:

1. **Text conditioning**: a frozen text encoder (e.g. CLIP) turns a prompt into embeddings, which are injected into the UNet via **cross-attention** layers (instead of only self-attention).
2. **Classifier-free guidance**: train the UNet to also handle an unconditional ("null prompt") input, then at sampling time extrapolate away from the unconditional prediction toward the conditional one — this is what makes prompts actually "grip" the output strongly.
3. **A much larger, more carefully trained autoencoder** (trained on billions of natural images, often with an added adversarial/perceptual loss) and a much larger UNet (with many more attention blocks), trained for far longer on far more data.
4. **Better samplers** (DDIM, DPM-Solver, etc.) that reach good samples in 20-50 steps instead of hundreds.

Everything else — the noise scheduler, the "predict-the-noise" training objective, the reverse diffusion sampling loop — is exactly what you already have here.